In [2]:
import pandas as pd
import numpy as np
import re
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
df=pd.read_csv('twitter_dataset.csv')
df=df.dropna(subset=['Text'])

In [5]:
def clean_tweet(tweet):
    tweet=re.sub(r'http\S+','',tweet)
    tweet=re.sub(r'\@\w+','',tweet)
    return tweet

In [6]:
df['cleaned_tweet']=df['Text'].apply(clean_tweet)
tokenized_tweets=[simple_preprocess(tweet) for tweet in df['cleaned_tweet']]
original_tweets=df['Text'].tolist()

In [7]:
w2v_model=Word2Vec(sentences=tokenized_tweets,vector_size=100,window=5,min_count=2,sg=0)

In [11]:
def get_document_embedding(doc_words,model):
    valid_words=[word for word in doc_words if word in model.wv]
    if not valid_words:
        return np.zeros(model.vector_size)
    return np.mean(model.wv[valid_words],axis=0)


In [12]:
doc_vectors=np.array([get_document_embedding(tweet,w2v_model) for tweet in tokenized_tweets])

In [15]:
def recommend(user_input):
    words = simple_preprocess(user_input)
    user_vector = get_document_embedding(words, w2v_model).reshape(1, -1)
    similarity = cosine_similarity(user_vector, doc_vectors)[0]
    
    top_indices = similarity.argsort()[-3:][::-1]
    
    for i in top_indices:
        print("Tweet:", original_tweets[i])
        print("Similarity:", similarity[i])
        print()
recommend("technology and new smartphones")

Tweet: Easy individual behavior would. Reality tend sure. Suggest whole woman crime officer eight voice. Century ability gas set argue end.
Inside character five. Compare even that wrong figure.
Similarity: 0.9996108

Tweet: Yard political over treatment as seat. Simply like and about and either dream.
Hit key partner. Network five left easy. Either finish civil above food road.
Class someone nearly too peace.
Similarity: 0.99959946

Tweet: Itself this nothing peace season finally wonder number. Possible effort take because cultural record.
Simple market television large with everyone. Provide information voice seven. Get reveal project how.
Effect out future Mr measure. Any do someone every.
Similarity: 0.99959296

